In [6]:
import catboost as cb
import numpy as np
import pandas as pd
from catboost import Pool, datasets
from catboost import *
from sklearn.model_selection import train_test_split

In [4]:
train_df, test_df = datasets.amazon()
train_df.head()

,ACTION,RESOURCE,MGR_ID,ROLE_ROLLUP_1,ROLE_ROLLUP_2,ROLE_DEPTNAME,ROLE_TITLE,ROLE_FAMILY_DESC,ROLE_FAMILY,ROLE_CODE
0,1,39353,85475,117961,118300,123472,117905,117906,290919,117908
1,1,17183,1540,117961,118343,123125,118536,118536,308574,118539
2,1,36724,14457,118219,118220,117884,117879,267952,19721,117880
3,1,36135,5396,117961,118343,119993,118321,240983,290919,118322
4,1,42680,5905,117929,117930,119569,119323,123932,19793,119325


In [9]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32769 entries, 0 to 32768
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   ACTION            32769 non-null  int64
 1   RESOURCE          32769 non-null  int64
 2   MGR_ID            32769 non-null  int64
 3   ROLE_ROLLUP_1     32769 non-null  int64
 4   ROLE_ROLLUP_2     32769 non-null  int64
 5   ROLE_DEPTNAME     32769 non-null  int64
 6   ROLE_TITLE        32769 non-null  int64
 7   ROLE_FAMILY_DESC  32769 non-null  int64
 8   ROLE_FAMILY       32769 non-null  int64
 9   ROLE_CODE         32769 non-null  int64
dtypes: int64(10)
memory usage: 2.5 MB


In [7]:
X, y = train_df.drop('ACTION', axis = 1), train_df['ACTION']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
y.value_counts()

ACTION
1    30872
0     1897
Name: count, dtype: int64

In [11]:
cat_features = list(range(X.shape[1]))
cat_features

[0, 1, 2, 3, 4, 5, 6, 7, 8]

In [ ]:
os.getcwd()

In [18]:
import os
DATA_SET_DIR = os.path.join(os.getcwd(),'amazon')

if not os.path.exists(DATA_SET_DIR):
    os.makedirs(DATA_SET_DIR)

train_df.to_csv(os.path.join(DATA_SET_DIR, 'train.tsv'), 
                index=False, sep='\t', header=False)


test_df.to_csv(os.path.join(DATA_SET_DIR, 'test.tsv'), 
              index=False, sep='\t', header=False)


train_df.to_csv(os.path.join(DATA_SET_DIR, 'train.csv'),
                index=False, sep=',', header=True)


test_df.to_csv(os.path.join(DATA_SET_DIR, 'test.csv'),
              index=False, sep=',', header=True)

In [19]:
!head amazon/train.csv

ACTION,RESOURCE,MGR_ID,ROLE_ROLLUP_1,ROLE_ROLLUP_2,ROLE_DEPTNAME,ROLE_TITLE,ROLE_FAMILY_DESC,ROLE_FAMILY,ROLE_CODE
1,39353,85475,117961,118300,123472,117905,117906,290919,117908
1,17183,1540,117961,118343,123125,118536,118536,308574,118539
1,36724,14457,118219,118220,117884,117879,267952,19721,117880
1,36135,5396,117961,118343,119993,118321,240983,290919,118322
1,42680,5905,117929,117930,119569,119323,123932,19793,119325
0,45333,14561,117951,117952,118008,118568,118568,19721,118570
1,25993,17227,117961,118343,123476,118980,301534,118295,118982
1,19666,4209,117961,117969,118910,126820,269034,118638,126822
1,31246,783,117961,118413,120584,128230,302830,4673,128231


In [36]:
from catboost.utils import create_cd

feature_names = dict()
cat_features = list()

for column, name in enumerate(train_df):
    if column == 0:
        continue
    feature_names[column] = name
    cat_features.append(column)
# feature_names, cat_features
create_cd(
    label=0,
    cat_features=cat_features,
    feature_names=feature_names,
    output_path=os.path.join(DATA_SET_DIR, 'train.cd')
)

In [37]:
!head amazon/train.cd

0	Label	
1	Categ	RESOURCE
2	Categ	MGR_ID
3	Categ	ROLE_ROLLUP_1
4	Categ	ROLE_ROLLUP_2
5	Categ	ROLE_DEPTNAME
6	Categ	ROLE_TITLE
7	Categ	ROLE_FAMILY_DESC
8	Categ	ROLE_FAMILY
9	Categ	ROLE_CODE


In [ ]:
pool1 = Pool(
    X, y
)

In [43]:
class Solution:
    def get_minimizer(self, iterations: int, learning_rate: float, init: int) -> float:
        step = init
        for iter in range(iterations):
            step -= learning_rate*2*step
        return round(step, 5)
    

In [44]:
iterations = 0
learning_rate = 0.01
init = 5
iterations = 10
learning_rate = 0.01
init = 5
Solution().get_minimizer(iterations, learning_rate, init)

4.08536

In [50]:
mat = [[3,3,1,1],[2,2,1,2],[1,1,1,2]]
m, n = len(mat), len(mat[0])

num_arrays = m + n - 1
sorted_mat = [[] for _ in range(num_arrays)]
num_arrays, sorted_mat

(6, [[], [], [], [], [], []])

In [87]:
banknotes = {banknot : 0 for banknot in [20, 50, 100, 200, 500]}
banknotes[100] = 1
banknotes[200] = 2
banknotes[500] = 1
banknotes

{20: 0, 50: 0, 100: 1, 200: 2, 500: 1}

In [88]:
remain = 600
takes = [0] * 5
for i, banknot in enumerate(reversed(banknotes.keys()), 1):
    used = min(remain // banknot, banknotes[banknot])
    
    remain -= banknot * used
    banknotes[banknot] -= used
    takes[-i] = used
    print(banknot, banknotes[banknot], used, banknotes, -i)
    if remain == 0:
        break
        # return takes

takes, banknotes

500 0 1 {20: 0, 50: 0, 100: 1, 200: 2, 500: 0} -1
200 2 0 {20: 0, 50: 0, 100: 1, 200: 2, 500: 0} -2
100 0 1 {20: 0, 50: 0, 100: 0, 200: 2, 500: 0} -3


([0, 0, 1, 0, 1], {20: 0, 50: 0, 100: 0, 200: 2, 500: 0})